# Dice vs no-Dice: test metrics analysis

This notebook compares test CSV metrics between two experiments (no-Dice vs Dice) and helps diagnose why CSI changes.
Adjust the experiment names and horizons to match your setup.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_rows", 200)

In [ ]:
base = Path("CSV_XP")
csvs = sorted(base.glob("Experience*.csv"))
csvs

In [ ]:
def load_csv(path):
    df = pd.read_csv(path)
    df["epoch"] = df["epoch"].astype(int)
    df["horizon_steps"] = df["horizon_steps"].astype(int)
    df["exp"] = path.stem
    return df

all_df = pd.concat([load_csv(p) for p in csvs], ignore_index=True)
all_df.head()

## Select experiments to compare

In [ ]:
exp_nodice = "Experience4"
exp_dice = "Experience5"
horizons = [6, 12, 24]  # 3h, 6h, 12h if dt=30min

df_nodice = all_df[all_df["exp"].eq(exp_nodice)].copy()
df_dice = all_df[all_df["exp"].eq(exp_dice)].copy()
df_nodice.head(), df_dice.head()

## Best CSI checkpoint per horizon

In [ ]:
def best_by_csi(df, horizons):
    rows = []
    for h in horizons:
        sub = df[df["horizon_steps"].eq(h)]
        if sub.empty:
            continue
        best = sub.loc[sub["csi"].idxmax()]
        rows.append({
            "horizon_steps": h,
            "epoch": int(best["epoch"]),
            "csi": float(best["csi"]),
            "mse_h": float(best["mse_h"]),
            "mse_u": float(best["mse_u"]),
            "mse_v": float(best["mse_v"]),
        })
    return pd.DataFrame(rows)

best_nodice = best_by_csi(df_nodice, horizons)
best_dice = best_by_csi(df_dice, horizons)
best_nodice, best_dice

## Align epochs and compute deltas (Dice - NoDice)

In [ ]:
merged = df_nodice.merge(
    df_dice,
    on=["epoch", "horizon_steps"],
    suffixes=("_nodice", "_dice"),
)
merged["dcsi"] = merged["csi_dice"] - merged["csi_nodice"]
merged["dmse_h"] = merged["mse_h_dice"] - merged["mse_h_nodice"]
merged["dmse_u"] = merged["mse_u_dice"] - merged["mse_u_nodice"]
merged["dmse_v"] = merged["mse_v_dice"] - merged["mse_v_nodice"]

merged[merged["horizon_steps"].isin(horizons)].groupby("horizon_steps")[
    ["dcsi", "dmse_h", "dmse_u", "dmse_v"]
].agg(["mean", "min", "max"])

## Plot CSI and MSE across epochs

In [ ]:
def plot_metric(metric, horizons):
    fig, axes = plt.subplots(1, len(horizons), figsize=(5 * len(horizons), 3), sharey=False)
    if len(horizons) == 1:
        axes = [axes]
    for ax, h in zip(axes, horizons):
        a = df_nodice[df_nodice["horizon_steps"].eq(h)].sort_values("epoch")
        b = df_dice[df_dice["horizon_steps"].eq(h)].sort_values("epoch")
        ax.plot(a["epoch"], a[metric], marker="o", linewidth=2, label=exp_nodice)
        ax.plot(b["epoch"], b[metric], marker="o", linewidth=2, label=exp_dice)
        ax.set_title(f"{metric} @ h={h}")
        ax.set_xlabel("epoch")
        ax.grid(True, alpha=0.3)
    axes[0].legend()
    plt.tight_layout()
    plt.show()

plot_metric("csi", horizons)
plot_metric("mse_h", horizons)

## Delta scatter: do CSI gains come with MSE changes?

In [ ]:
fig, axes = plt.subplots(1, len(horizons), figsize=(5 * len(horizons), 3), sharey=True)
if len(horizons) == 1:
    axes = [axes]
for ax, h in zip(axes, horizons):
    sub = merged[merged["horizon_steps"].eq(h)]
    ax.scatter(sub["dmse_h"], sub["dcsi"], alpha=0.8)
    ax.axhline(0, color="black", linewidth=1)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_title(f"Delta CSI vs Delta MSE_h @ h={h}")
    ax.set_xlabel("dmse_h (dice - nodice)")
axes[0].set_ylabel("dcsi (dice - nodice)")
plt.tight_layout()
plt.show()

## Correlation between CSI and MSE inside each experiment

In [ ]:
def corr_table(df, horizons):
    rows = []
    for h in horizons:
        sub = df[df["horizon_steps"].eq(h)]
        if len(sub) < 2:
            continue
        rows.append({
            "horizon_steps": h,
            "corr_csi_mse_h": sub["csi"].corr(sub["mse_h"]),
            "corr_csi_mse_u": sub["csi"].corr(sub["mse_u"]),
            "corr_csi_mse_v": sub["csi"].corr(sub["mse_v"]),
        })
    return pd.DataFrame(rows)

corr_nodice = corr_table(df_nodice, horizons)
corr_dice = corr_table(df_dice, horizons)
corr_nodice, corr_dice

## Best/worst epochs for CSI delta

In [ ]:
for h in horizons:
    sub = merged[merged["horizon_steps"].eq(h)]
    if sub.empty:
        continue
    best = sub.loc[sub["dcsi"].idxmax()]
    worst = sub.loc[sub["dcsi"].idxmin()]
    print("h=", h)
    print(" best epoch", int(best["epoch"]), "dcsi", best["dcsi"], "dmse_h", best["dmse_h"])
    print(" worst epoch", int(worst["epoch"]), "dcsi", worst["dcsi"], "dmse_h", worst["dmse_h"])
    print()

## Optional diagnostics if you have raw predictions

To test the "over-wetting" hypothesis, compute:

- wet fraction: mean( h > eps_front ) for pred vs gt
- bias in water height: mean(h_pred - h_gt)
- false positives on wet mask: mean( (h_pred>eps) & (h_gt<=eps) )

If Dice increases wet fraction and bias, it can reduce CSI even if MSE improves locally.